# 10 — Per-speaker standardisation and within-block orthogonalisation

**Stage question.** Do both standardisation modes behave as registered, does the held-out-speaker rule exclude the target and its context turns, and does orthogonalisation remove within-block correlation?

| | |
|---|---|
| **Purpose** | Assemble the analysis set and its arrays (Y, A, S, Z) with every transform fitted on training rows, and show the leakage-critical rules on concrete cases. |
| **Input** | Outputs of notebooks 07-09. |
| **Processing** | analysis set → training speaker statistics → standardisation (two modes) → ZCA-cor orthogonalisation fitted on training targets → PCA-64 → Z. |
| **Output** | `results/standardisation/` checks and the exclusion table. |
| **Scientific assumptions** | Levitan and Hirschberg's per-speaker norming; minimum turns per speaker (`src/settings.py`); ZCA-cor whitening. |
| **Failure modes** | Held-out targets exactly 0/1 (would mean test rows fed their own statistics; the assembly raises). |
| **What to look at** | The standardisation check, the exclusion table, the block covariance. |

**Terms used in this notebook.**

- **Training-speaker mode** — statistics from the speaker's training turns.
- **Held-out-speaker mode** — statistics from the speaker's other turns in the same session, without the target turn and the turns used as its context.
- **Orthogonalisation** — a linear transform that makes a block's dimensions uncorrelated.

> **Before running.** The minimum number of turns per speaker is set in `src/settings.py`.

In [2]:
# Runtime setup — identical in every notebook.
# Colab: mounts Google Drive and uses MyDrive/coda (a copy of the repository) as
#        the project root, then installs the pinned measurement stack.
# Local: uses the repository checkout.
# Every path used below is relative to that root, so Drive and the repository
# have exactly the same layout and outputs can be copied back unchanged.
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    ROOT = Path(os.environ.get('CODA_ROOT', '/content/drive/MyDrive/coda'))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(ROOT / 'requirements-colab.txt')], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

os.environ['CODA_ROOT'] = str(ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.runtime import runtime_report
REPORT = runtime_report()

Mounted at /content/drive
runtime        colab   python 3.13.16   cpus 2
memory         12.3 GB free of 13.6 GB
gpu            none (not needed for this stage unless stated)
project disk   11.6 GB free   root = /content/drive/MyDrive/coda
scratch disk   93.4 GB free   (audio lives here briefly)
code           fingerprint 9b5146f3ab46b0fc
artifacts      manifest=True, metadata=True, transcripts=True, turns=True, audio_pass_sessions=293, channel_qc=True


**Runtime check.** The cell above prints free disk, memory, CPU, GPU, the code fingerprint and which artifacts already exist. The fingerprint identifies the exact source and configuration that ran and is written into every run record.

In [3]:
STAGE = 'sample'
STEPS = ['standardisation']

## Settings

The method settings come from `src/settings.py` (class `Settings`).

In [4]:
from src.settings import Settings
SETTINGS = Settings()

## 1. Assemble the analysis data

In [5]:
import pandas as pd
from src.paths import paths
from src.experiments.stages import analysis_data
P = paths()
data, turns, prosody, split = analysis_data(P, STAGE, 'speaker_disjoint', SETTINGS)
import json
print(json.dumps(data.provenance()['pairs_by_split'], indent=1))
pd.DataFrame(data.checks['exclusions'])

/usr/local/lib/python3.13/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/content/drive/MyDrive/coda/src/data/dataset.py:180: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False).astype(bool),


{
 "train": 37600,
 "test": 12107,
 "val": 6922
}


,test,train,val,All
included,12107,37600,6922,56629
simultaneous_onset,40,160,19,219
reply_not_usable,4613,13459,1347,19419
partner_not_usable,3408,10172,1083,14663
no_self_previous,389,1201,209,1799
self_previous_not_usable,2460,7311,829,10600
no_n2_candidate,469,1899,296,2664
All,23486,71802,10705,105993


**Reading it.** The table counts every candidate pair by exclusion reason and partition; rows add up to all candidates. One common set serves every condition, so droppage cannot differ between conditions.

## 2. The near-but-not-exactly check

In [6]:
data.checks['standardisation_heldout_targets']

{'mean': [0.027123299006708134,
  -0.006034377516213597,
  0.0022912139346395006,
  0.02008020737502378,
  -0.027474140248287388,
  -0.011500752538518707,
  -0.05312470979982236,
  -0.04402435962681731],
 'sd': [1.022206899910401,
  1.0224017707032516,
  1.038365949775207,
  1.035048153701765,
  1.0106169321454432,
  1.0182017424795422,
  0.9698254688608801,
  1.063991821190088],
 'exactly_zero_one': False,
 'n': 19029}

**Reading it.** Held-out targets should have means near 0 and SDs near 1, but not exactly, because their statistics come from other turns. Exactly 0 and 1 would show the target turns had entered their own statistics; the assembly would have raised.

## 3. The leave-out rule on one concrete pair

In [7]:
import numpy as np
from src.targets.normalise import standardise_rows, MODE_HELD_OUT
from src.experiments.pipeline import DIMS
row = data.pairs[data.split == 'test'].iloc[0]
excl = [[row['reply_turn_id'], row['partner_turn_id'], row['self_previous_turn_id']]]
base = standardise_rows([row['reply_turn_id']], [MODE_HELD_OUT], excl, prosody, DIMS, None, 2)
bumped = prosody.copy()
bumped.loc[bumped['turn_id'] == row['self_previous_turn_id'], DIMS] += 100
same = standardise_rows([row['reply_turn_id']], [MODE_HELD_OUT], excl, bumped, DIMS, None, 2)
# positive control: alter an ordinary turn of the same speaker and session instead
r = prosody.set_index('turn_id').loc[row['reply_turn_id']]
other = prosody[(prosody['speaker'] == r['speaker']) & (prosody['session_key'] == r['session_key'])
                & prosody['usable'].astype(bool) & ~prosody['turn_id'].isin(excl[0])]['turn_id'].iloc[0]
moved_p = prosody.copy()
moved_p.loc[moved_p['turn_id'] == other, DIMS] += 100
moved = standardise_rows([row['reply_turn_id']], [MODE_HELD_OUT], excl, moved_p, DIMS, None, 2)
d_excl = float(np.abs(base[DIMS].values - same[DIMS].values).max())
d_other = float(np.abs(base[DIMS].values - moved[DIMS].values).max())
print(f'max change after altering the EXCLUDED context turn: {d_excl:.1e}  (rounding only if < 1e-9)')
print(f'max change after altering an ORDINARY turn:          {d_other:.2f}  (must be large)')
print('leave-out rule holds:', d_excl < 1e-9 and d_other > 1e-3)


max change after altering the EXCLUDED context turn: 4.1e-13  (rounding only if < 1e-9)
max change after altering an ORDINARY turn:          1.30  (must be large)
leave-out rule holds: True


**Reading it.** The replier's own previous turn is part of this pair's context, so changing it must not move the standardised target. The printed flag shows the rule acting on the corpus.

## 4. Within-speaker serial dependence (diagnostic D-AC)

In [8]:
from src.analysis.serial_dependence import within_speaker_lag1
lag1 = within_speaker_lag1(data)
display(lag1.round(3))
lag1.to_csv(P.results / 'eda' / f'serial_dependence_{STAGE}.csv', index=False)

,dimension,n_pairs,r_raw,r_residual_on_Z_content
0,mean_log_f0,37600,0.243,0.237
1,log_f0_range_p95_p5,37600,0.075,0.068
2,log_f0_slope_final_300ms,37600,0.001,0.000
3,mean_intensity,37600,0.243,0.209
4,intensity_range,37600,0.036,0.024
5,articulation_rate,37600,0.101,0.065
6,pause_ratio,37600,0.134,0.055
7,longest_internal_pause,37600,0.096,0.032


**Reading it.** Each row is the correlation, on training pairs only, between a speaker's standardised delivery and their own previous turn. The last column is that correlation after both are residualised on Z_content, the residual J3 resamples. Descriptive only: no partner-to-reply quantity is computed. Place the residual column on the power-projection sensitivity table (`src/experiments/power_projection.py`). Near 0 means the optimistic MDE (about 0.0005) is the relevant scenario; near 0.6 means about 0.0015; near 0.9 means about 0.0044. Nothing about the analysis is changed by this result. J3 uses order-preserving resampling in every case.